In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install vectorbt

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install TA-lib

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install --upgrade statsmodels

In [5]:
!pip install fredapi pandas

In [6]:
import scipy
import statsmodels.api as sm

In [7]:
import yfinance as yf
import numpy as np
import pandas as pd
import talib
from datetime import datetime

In [8]:
# import data from ticker 

ydf = yf.download("NVDA",
                  start="2015-01-01",
                  end="2026-04-30",
                  progress=False)

In [9]:
print(f"Downloaded {len(ydf)} rows of data.")

Downloaded 2847 rows of data.


In [10]:
ydf

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000
...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400


In [11]:
ydf.columns

MultiIndex([( 'Close', 'NVDA'),
            (  'High', 'NVDA'),
            (   'Low', 'NVDA'),
            (  'Open', 'NVDA'),
            ('Volume', 'NVDA')],
           names=['Price', 'Ticker'])

In [12]:
# Create column for the length of the candle body and daily price action (Total candle range)
ydf['Candle_Body'] = (ydf[('Close', 'NVDA')] - ydf[('Open', 'NVDA')]).abs()
ydf['Total price action'] = (ydf[('High', 'NVDA')] - ydf[('Low', 'NVDA')])

# Create a column to determine if Candle is bullish, bearish or a doji
doji = abs(ydf['Close', 'NVDA'] - ydf['Open', 'NVDA']) <= (0.10 * (ydf['High', 'NVDA'] - ydf['Close', 'NVDA']))
ydf['Bullish/Bearish'] = np.where(
    doji,
    'Doji',
    np.where((ydf['Close', 'NVDA'] > ydf['Open', 'NVDA']), 'Bullish', 'Bearish')
)

# Create column for candle body percentage by dividing the candle body by the total price action of the day
ydf['Body %'] = (ydf['Candle_Body'] / ydf['Total price action']) * 100

# Change total price action column to Candle_range 

ydf = ydf.rename(columns={'Total price action': 'Candle_Range'})

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,
Date,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899
...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032


In [13]:
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,
Date,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899
...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032


In [14]:
#Add 5, 10, 20, 50, and 200 day EMA Columns 

ydf['5_EMA'] = ydf['Close', 'NVDA'].ewm(span=5, adjust=False).mean()
ydf['10_EMA'] = ydf['Close', 'NVDA'].ewm(span=10, adjust=False).mean()
ydf['20_EMA'] = ydf['Close', 'NVDA'].ewm(span=20, adjust=False).mean()
ydf['50_EMA'] = ydf['Close', 'NVDA'].ewm(span=50, adjust=False).mean()
ydf['200_EMA'] = ydf['Close', 'NVDA'].ewm(span=20, adjust=False).mean()

ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,
Date,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000,0.482423,0.482423,0.482423,0.482423,0.482423
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800,0.479707,0.480942,0.481647,0.482104,0.481647
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920,0.473103,0.477115,0.479575,0.481233,0.479575
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879,0.468301,0.473766,0.477587,0.480349,0.477587
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899,0.470852,0.474164,0.477431,0.480176,0.477431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739,199.743683,196.414476,190.964749,186.173519,190.964749
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032,202.504969,198.525943,192.589777,187.030540,192.589777


In [15]:
import talib

In [16]:
ydf['RSI'] = talib.RSI(ydf['Close', 'NVDA'])
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,10_EMA,20_EMA,50_EMA,200_EMA,RSI
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,,,,,
Date,,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000,0.482423,0.482423,0.482423,0.482423,0.482423,NaN
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800,0.479707,0.480942,0.481647,0.482104,0.481647,NaN
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920,0.473103,0.477115,0.479575,0.481233,0.479575,NaN
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879,0.468301,0.473766,0.477587,0.480349,0.477587,NaN
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899,0.470852,0.474164,0.477431,0.480176,0.477431,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739,199.743683,196.414476,190.964749,186.173519,190.964749,64.645031
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032,202.504969,198.525943,192.589777,187.030540,192.589777,71.498288


In [17]:
import vectorbt as vbt

In [18]:
#Calculate the upper wick by subtracking the high from either the open or close. Use max function and axis=1 to choose the option closest to the upperwick
ydf['Upper_wick'] = abs(ydf['High', 'NVDA'] - ydf[[('Open', 'NVDA'), ('Close', 'NVDA')]].max(axis=1))
# Caculate the same for the lower wick using the min function with axis=1 so it chooses the value closest to the lower wick between the open and close
ydf['Lower_wick'] = abs(ydf['Low', 'NVDA'] - ydf[[('Open', 'NVDA'), ('Close', 'NVDA')]].min(axis=1))

# Calculate wick percentage by dividing the absolute value of the upper or lower wick by the candle ranage (total candle length)
ydf['Upper_wick_percentage'] = abs(ydf['Upper_wick'] / ydf['Candle_Range']) * 100
ydf['Lower_wick_percentage'] = abs(ydf['Lower_wick'] / ydf['Candle_Range']) * 100

# Calculate Wick to body ratio by dividing the sum of the wicks by the candle body
ydf['Wick_to_body_ratio'] = (ydf['Lower_wick'] + ydf['Upper_wick']) / ydf['Candle_Body']

# Calsulate the gap size 
ydf['gap_size'] = ydf['Open', 'NVDA'] - ydf['Close', 'NVDA'].shift(1)
# Calculate gap percentage 
ydf['gap_percentage'] = (ydf['gap_size'] / ydf['Close', 'NVDA'].shift(1)) * 100

# Caluclate distance from EMA distnace Percentage 
ydf['5EMA_touched'] = np.where((ydf['5_EMA'] >= ydf['Low', 'NVDA']) & (ydf['5_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['10EMA_touched'] = np.where((ydf['10_EMA'] >= ydf['Low', 'NVDA']) & (ydf['10_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['20EMA_touched'] = np.where((ydf['20_EMA'] >= ydf['Low', 'NVDA']) & (ydf['20_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['50EMA_touched'] = np.where((ydf['50_EMA'] >= ydf['Low', 'NVDA']) & (ydf['50_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf['200EMA_touched'] = np.where((ydf['200_EMA'] >= ydf['Low', 'NVDA']) & (ydf['200_EMA'] <= ydf['High', 'NVDA']), 'True', 'False')
ydf

Price,Close,High,Low,Open,Volume,Candle_Body,Candle_Range,Bullish/Bearish,Body %,5_EMA,...,Upper_wick_percentage,Lower_wick_percentage,Wick_to_body_ratio,gap_size,gap_percentage,5EMA_touched,10EMA_touched,20EMA_touched,50EMA_touched,200EMA_touched
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,,,,,,...,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,0.000000,0.011264,Doji,0.000000,0.482423,...,31.915002,68.084998,inf,NaN,NaN,True,True,True,True,True
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,0.008148,0.011743,Bearish,69.387800,0.479707,...,12.244977,18.367223,0.441176,-7.587153e-08,-0.000016,True,True,True,True,True
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,0.015098,0.016057,Bearish,94.029920,0.473103,...,2.985040,2.985040,0.063491,7.190050e-04,0.151601,True,False,False,False,False
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,0.004553,0.010065,Bearish,45.237879,0.468301,...,40.476326,14.285795,1.210537,3.355084e-03,0.729532,False,False,False,False,False
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,0.011983,0.015098,Bullish,79.364899,0.470852,...,19.047727,1.587374,0.260003,5.272421e-03,1.149434,True,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,2.816724,6.602306,Bearish,42.662739,199.743683,...,20.726097,36.611164,1.343966,-3.994972e-02,-0.019751,True,False,False,False,False
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,8.300323,11.127031,Bullish,74.596032,202.504969,...,24.057386,1.346581,0.340554,3.196261e-01,0.160288,True,False,False,False,False


In [20]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [32]:
pip install cpi

   ---------------------------------------- 0.0/18.1 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/18.1 MB 18.5 MB/s eta 0:00:01
   ----------------- ---------------------- 8.1/18.1 MB 22.6 MB/s eta 0:00:01
   -------------------------------- ------- 14.9/18.1 MB 27.4 MB/s eta 0:00:01
   ---------------------------------------- 18.1/18.1 MB 26.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [47]:
pip install --upgrade cpi

Note: you may need to restart the kernel to use updated packages.


In [48]:
import cpi

In [49]:
df_cpi = cpi_series.to_dataframe()
df_cpi

,series_id,year,date,value,period_id,period_code,period_abbreviation,period_name,period_month,period_type
0,CUUR0000SA0,1997,1997-01-01,159.1,M01,M01,JAN,January,1,monthly
1,CUUR0000SA0,1997,1997-02-01,159.6,M02,M02,FEB,February,2,monthly
2,CUUR0000SA0,1997,1997-03-01,160.0,M03,M03,MAR,March,3,monthly
3,CUUR0000SA0,1997,1997-04-01,160.2,M04,M04,APR,April,4,monthly
4,CUUR0000SA0,1997,1997-05-01,160.1,M05,M05,MAY,May,5,monthly
...,...,...,...,...,...,...,...,...,...,...
1468,CUUR0000SA0,1996,1996-09-01,157.8,M09,M09,SEP,September,9,monthly
1469,CUUR0000SA0,1996,1996-10-01,158.3,M10,M10,OCT,October,10,monthly
1470,CUUR0000SA0,1996,1996-11-01,158.6,M11,M11,NOV,November,11,monthly
1471,CUUR0000SA0,1996,1996-12-01,158.6,M12,M12,DEC,December,12,monthly


In [22]:
ndf = yf.download("NVDA",
                  start="2015-01-01",
                  end="2026-04-30",
                  progress=False)
ndf

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000
...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400


In [23]:
ndf['5_EMA'] = talib.EMA(ndf['Close', 'NVDA'].to_numpy(), timeperiod=20)
ndf

Price,Close,High,Low,Open,Volume,5_EMA
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA,
Date,,,,,,
2015-01-02,0.482423,0.486018,0.474754,0.482423,113680000,NaN
2015-01-05,0.474275,0.483861,0.472118,0.482423,197952000,NaN
2015-01-06,0.459896,0.475473,0.459416,0.474994,197764000,NaN
2015-01-07,0.458697,0.467325,0.457259,0.463251,321808000,NaN
2015-01-08,0.475952,0.478828,0.463730,0.463970,283780000,NaN
...,...,...,...,...,...,...
2026-04-23,199.407593,203.592718,196.990412,202.224317,113561800,190.964749
2026-04-24,208.027542,210.704415,199.577384,199.727219,214134400,192.589777


In [24]:
ydf['20)

SyntaxError: unterminated string literal (detected at line 1) (4034197693.py, line 1)

In [ ]:
this can be used to compare columns based on differnt days 

ydf.shift()

In [ ]:
ydf['Low']

In [ ]:
pip install nasdaq-data-link

In [ ]:
import nasdaqdatalink as ndl

In [ ]:
My_Nas_Key = "bp9V2Fn_MCJ8xEUxYS8p"

ndl.ApiConfig.api_key = My_Nas_Key

In [ ]:
ndf